# CLIP patches + standard decoder — TRAIN smoke and benchmark

GPU **T4×2**, Internet ON, Save & Run All. Attach **visual-cache** and **vuthetam/mscoco-2014**. No detector, ROI134, bank, prompt or API token.

FrozenCLIP-B16 raw197tokens dropsCLS;196patch→Linear/LayerNorm512→standard4-layer causal TransformerDecoder→vocab. Fresh initialization; this is NOT old plain/CONTROL checkpoint continuation.

Only256fixedTRAINimages,200XEupdates/global32; no VALID/TEST/CIDEr/SCST/reranking. Check causal mask, image-conditioning, global-token DDP gradient parity and small-pool learning. Measure memory/throughput; small-pool timing does not guarantee full-data speed.

Latest reported budget30hGPU. Parent45min STOPcap is not ETA or billed quota. Full epoch budget/dataIO will be chosen AFTER this result. No target110CIDEr promise.

Send `clip_patch_baseline_smoke_v1_review.zip`. No full baseline trained yet; do not put these TRAIN losses into the quality comparison table.


In [ ]:
import hashlib,json,os,queue,signal,subprocess,sys,tempfile,threading,time,traceback,urllib.request,zipfile,shutil
from pathlib import Path
REVISION='e6df0f1a298356ee86e49ae4d736dc2a23fabda9'
MAPPING={'experiments/clip_patch_baseline/patch_data.py': 'patch_data.py', 'experiments/clip_patch_baseline/patch_model.py': 'patch_model.py', 'experiments/clip_patch_baseline/run_smoke.py': 'run_smoke.py', 'experiments/clip_patch_baseline/test_patch_data.py': 'test_patch_data.py', 'experiments/clip_patch_baseline/PROTOCOL.md': 'PROTOCOL.md', 'experiments/csa_paper/data.py': 'data.py', 'experiments/csa_paper/real_bank_data.py': 'real_bank_data.py', 'experiments/csa_paper/cuda_runtime_guard.py': 'cuda_runtime_guard.py'}
HASHES={'experiments/clip_patch_baseline/patch_data.py': '3706ffec9c128c8fe12c4f21ba071939268fdd526ed614034cca8dfec55674d5', 'experiments/clip_patch_baseline/patch_model.py': '2d02bfb62627b33caf178a25fbcf2fcae4cd7429848944ecb53a8fb372f25ac9', 'experiments/clip_patch_baseline/run_smoke.py': '540767d0423dc3ae56aae472977ba5d04d5881eac054b4d464bb9b907dcdbe70', 'experiments/clip_patch_baseline/test_patch_data.py': 'f62af4d1c88fc72668c8ce877ef04985a5c093702b15c73db42866a4e85bb3a0', 'experiments/clip_patch_baseline/PROTOCOL.md': '749a49c66a73b9cbea5bb07f2a1e6c66ed34d3e7d38c6b5fbd3c92b1ba241ea6', 'experiments/csa_paper/data.py': '5a5bb451dc20184d3ea597d4dc8415a120583571f4e1a4dac75e29a306213a71', 'experiments/csa_paper/real_bank_data.py': '2c791f8dd856198ec1954a2edd1f9e46520d422f20eff07220e462fc2c9bd9a0', 'experiments/csa_paper/cuda_runtime_guard.py': '640b5debc81607e3504930e4042571636cc3fbd525a6d0e5eb45abdd141fb773'}
VISUAL_CACHE=Path('/kaggle/input/datasets/ducanh2403/visual-cache')
COCO_JSON=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
OUTPUT=Path('/kaggle/working/clip_patch_baseline_smoke_v1')
for p in (VISUAL_CACHE,COCO_JSON):
    if not p.exists():raise FileNotFoundError('Attach/correct input before GPU: '+str(p))
if OUTPUT.exists():raise RuntimeError('Preserve prior output; use fresh runtime/output path')
if min(shutil.disk_usage('/tmp').free,shutil.disk_usage('/kaggle/working').free)<2*1024**3:raise RuntimeError('Need2GiB free for small TRAIN stage')
OUTPUT.mkdir(parents=True);SOURCE=Path(tempfile.mkdtemp(prefix='clip-patch-baseline-',dir='/tmp'));LOG=OUTPUT/'notebook.log'
PARENT_DEADLINE=time.monotonic()+2700
os.environ['PYTHONPATH']=str(SOURCE);os.environ['OMP_NUM_THREADS']='1'
CONFIG=SOURCE/'config.json'
cfg=dict(output=str(OUTPUT),stage=str(SOURCE/'stage'),dataset_json=str(COCO_JSON),visual_cache=str(VISUAL_CACHE),updates=200,prepare_seconds=1200,worker_seconds=900)
CONFIG.write_text(json.dumps(cfg,indent=2));(OUTPUT/'protocol.json').write_text(json.dumps(cfg,indent=2))


In [ ]:
def run(args, timeout=1800):
    if time.monotonic()>=PARENT_DEADLINE:raise TimeoutError('Parent cap exceeded')
    timeout=min(timeout,PARENT_DEADLINE-time.monotonic())
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('Patch baseline active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('Smoke wall cap exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
complete=False
try:
    for name,destination in MAPPING.items():
        remaining=PARENT_DEADLINE-time.monotonic()
        if remaining<=0:raise TimeoutError('Source download parent cap exceeded')
        raw=urllib.request.urlopen('https://raw.githubusercontent.com/Supzxjee/Image_Captioning/'+REVISION+'/'+name,timeout=min(90,remaining)).read()
        if hashlib.sha256(raw).hexdigest()!=HASHES[name]:raise ValueError('Changed source: '+name)
        (SOURCE/destination).write_bytes(raw)
    (OUTPUT/'source_receipt.json').write_text(json.dumps(dict(commit=REVISION,sha256=HASHES),indent=2))
    run([sys.executable,'-u',SOURCE/'test_patch_data.py'],120)
    run([sys.executable,'-u',SOURCE/'cuda_runtime_guard.py','--output',OUTPUT],60)
    run([sys.executable,'-u',SOURCE/'run_smoke.py','prepare','--config',CONFIG],1200)
    cfg['worker_seconds']=min(900,PARENT_DEADLINE-time.monotonic()-120)
    if cfg['worker_seconds']<180:raise TimeoutError('Insufficient budget after staging; preserve inputs/logs')
    CONFIG.write_text(json.dumps(cfg,indent=2));(OUTPUT/'protocol.json').write_text(json.dumps(cfg,indent=2))
    run([sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2','--max_restarts=0',SOURCE/'run_smoke.py','worker','--config',CONFIG],cfg['worker_seconds']+30)
    complete=json.loads((OUTPUT/'result.json').read_text())['status']=='CLIP_PATCH_BASELINE_TRAIN_SMOKE_COMPLETE'
except BaseException:
    (OUTPUT/'failure.txt').write_text(traceback.format_exc());raise
finally:
    (OUTPUT/'run_status.json').write_text(json.dumps(dict(complete=complete,full_training_complete=False,test_used=False,validation_used=False)))
    review=OUTPUT.with_name(OUTPUT.name+'_review.zip')
    with zipfile.ZipFile(review,'w',zipfile.ZIP_DEFLATED) as z:
        for p in OUTPUT.rglob('*'):
            if p.is_file():z.write(p,str(p.relative_to(OUTPUT)))
    print('SEND REVIEW:',review,flush=True)
    print('TRAIN SMOKE COMPLETE; not full XE or CIDEr result.' if complete else 'INCOMPLETE; preserve output/logs, no automatic retry.',flush=True)
